# MLP Classifier Analysis

Aggregated DRIAMS (A+B+C+D) -- Top 3 drugs

Uses **MaldiDeepKit's `MaldiMLPClassifier`** (PyTorch MLP with optional sigmoid-gated attention).

**Approaches:** A) Baseline MLP  B) Regularised MLP + threshold tuned  C) Attention MLP + threshold tuned
**Preprocessing:** log1p + standardise (fit on train only)
**Splitting:** Species-stratified 70/15/15

In [ ]:
!pip install maldideepkit maldiamrkit --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldideepkit.attention.mlp import MaldiMLPClassifier
from maldiamrkit.evaluation import stratified_species_drug_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
                              ConfusionMatrixDisplay, RocCurveDisplay)
warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
try:
    get_ipython().run_line_magic('matplotlib', 'widget')
except: pass

In [ ]:
if IN_COLAB:
    DATA_ROOT = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet/Processed")
else:
    DATA_ROOT = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet/Processed")
OUT_DIR = Path("./results_mlp_aggregated")
OUT_DIR.mkdir(exist_ok=True)
print(f"Data: {DATA_ROOT}")
print(f"Output: {OUT_DIR.resolve()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
SITES = ["Proc_DRIAMS-A", "Proc_DRIAMS-B", "Proc_DRIAMS-C", "Proc_DRIAMS-D"]
BIN_COLS = [f"bin_{i}" for i in range(6000)]
DRUGS = ["Ciprofloxacin", "Amoxicillin-Clavulanic_acid", "Gentamicin"]

def load_aggregated_drug(drug_name):
    frames = []
    for site in SITES:
        p = DATA_ROOT / site / drug_name / "data.csv"
        if p.exists():
            df = pd.read_csv(p)
            df["site"] = site
            frames.append(df)
    df_all = pd.concat(frames, ignore_index=True)
    X = df_all[BIN_COLS].values.astype("float32")
    y = df_all["label"].values.astype(int)
    species = df_all["species"].values
    print(f"  {drug_name:35s}  {X.shape[0]:6d} samples  R={sum(y==1):5d}  S={sum(y==0):5d}")
    return X, y, species

def create_splits(X, y, species, train_size=0.70, test_size=0.15, seed=SEED):
    n = len(y)
    idx = np.arange(n)
    idx_trval, idx_test, _, _ = stratified_species_drug_split(
        idx.reshape(-1, 1), y, species=species, test_size=test_size, random_state=seed)
    idx_trval = idx_trval.flatten().astype(int)
    idx_test = idx_test.flatten().astype(int)
    X_trval, X_test = X[idx_trval], X[idx_test]
    y_trval, y_test = y[idx_trval], y[idx_test]
    sp_trval = species[idx_trval]
    val_frac = 0.15 / (1 - test_size)
    n2 = len(y_trval)
    idx2 = np.arange(n2)
    idx_train, idx_val, _, _ = stratified_species_drug_split(
        idx2.reshape(-1, 1), y_trval, species=sp_trval, test_size=val_frac, random_state=seed)
    idx_train = idx_train.flatten().astype(int)
    idx_val = idx_val.flatten().astype(int)
    X_train, X_val = X_trval[idx_train], X_trval[idx_val]
    y_train, y_val = y_trval[idx_train], y_trval[idx_val]
    return {"train": (X_train, y_train), "val": (X_val, y_val), "test": (X_test, y_test)}
print("Loading + splitting + preprocessing...")
preprocessed = {}
for drug in DRUGS:
    X, y, species = load_aggregated_drug(drug)
    spl = create_splits(X, y, species)
    state = fit_input_transform(spl["train"][0], "log1p+standardize")
    pp = {}
    for part in ["train", "val", "test"]:
        pp[part] = (apply_input_transform(spl[part][0], state), spl[part][1])
    preprocessed[drug] = pp
    print(f"  {drug}: train={pp['train'][0].shape[0]:5d}  val={pp['val'][0].shape[0]:5d}  test={pp['test'][0].shape[0]:5d}")
print("Done.")

---
## Drug 1: Ciprofloxacin (23,662 samples across all 4 sites)

In [ ]:
# Drug 1: Ciprofloxacin
DRUG = "Ciprofloxacin"
X_train, y_train = preprocessed[DRUG]["train"]
X_val,   y_val   = preprocessed[DRUG]["val"]
X_test,  y_test  = preprocessed[DRUG]["test"]
print(f"Ciprofloxacin -- Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# Approach A: Baseline MLP (no regularisation)
mlp_raw = MaldiMLPClassifier(
    hidden_dim=512, head_dims=(256, 128), use_attention=False,
    dropout_high=0.0, dropout_low=0.0, weight_decay=0.0,
    learning_rate=1e-4, batch_size=32, epochs=100,
    early_stopping_patience=10, val_fraction=0.1,
    input_transform="none", tune_threshold=False, random_state=SEED, verbose=True)
mlp_raw.fit(X_train, y_train)

results_a = {}
for name, X_s, y_s in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    preds = mlp_raw.predict(X_s)
    proba = mlp_raw.predict_proba(X_s)[:, 1]
    results_a[name] = {"BalAcc": balanced_accuracy_score(y_s, preds),
                       "F1": f1_score(y_s, preds, average="macro"),
                       "AUC": roc_auc_score(y_s, proba)}

print("\nApproach A -- Baseline MLP (no reg)")
for split, m in results_a.items():
    print(f"  {split:6s}  BalAcc={m['BalAcc']:.4f}  F1={m['F1']:.4f}  AUC={m['AUC']:.4f}")

### Approach B -- Grid Search (lr x dropout)

In [ ]:
# Approach B: Grid search lr x dropout (8x8 = 64 combos)
LR_GRID   = np.linspace(7.5e-5, 9.5e-5, 8)
DROP_GRID = np.linspace(0.5, 0.8, 8)
print(f"Coarse grid: {len(LR_GRID)} lr x {len(DROP_GRID)} dropout = {len(LR_GRID)*len(DROP_GRID)} combos")

grid_results = []
best_balacc = -1; best_lr = None; best_dh = None; best_dl = None

for lr in LR_GRID:
    for d in DROP_GRID:
        dh, dl = d, d / 2
        mlp_tmp = MaldiMLPClassifier(
            hidden_dim=512, head_dims=(256, 128), use_attention=False,
            dropout_high=dh, dropout_low=dl, weight_decay=1e-3,
            learning_rate=lr, batch_size=64, epochs=50,
            early_stopping_patience=10, warmup_epochs=10,
            val_fraction=0.1, use_sam=False,
            input_transform="none", tune_threshold=False, random_state=SEED, verbose=False)
        mlp_tmp.fit(X_train, y_train)
        val_balacc = balanced_accuracy_score(y_val, mlp_tmp.predict(X_val))
        val_auc = roc_auc_score(y_val, mlp_tmp.predict_proba(X_val)[:, 1])
        grid_results.append({"lr": lr, "dh": dh, "dl": dl,
                             "val_balacc": val_balacc, "val_auc": val_auc})
        mark = " *" if val_balacc > best_balacc else ""
        print(f"  lr={lr:.1e}  drop=({dh:.2f},{dl:.2f})  val_balacc={val_balacc:.4f}  val_auc={val_auc:.4f}{mark}")
        if val_balacc > best_balacc:
            best_balacc = val_balacc; best_lr = lr; best_dh = dh; best_dl = dl

print(f"\nBest coarse: lr={best_lr:.1e}  drop=({best_dh:.2f},{best_dl:.2f})  val_balacc={best_balacc:.4f}")

In [ ]:
# Fine grid around coarse best
LR_FACTOR = 2.0; DROP_WINDOW = 0.12
LR_GRID_FINE = np.logspace(np.log10(max(1e-6, best_lr / LR_FACTOR)),
                           np.log10(best_lr * LR_FACTOR), 5)
DROP_GRID_FINE = np.linspace(max(0.05, best_dh - DROP_WINDOW),
                              min(0.95, best_dh + DROP_WINDOW), 5)
print(f"Fine lr: {[f'{lr:.2e}' for lr in LR_GRID_FINE]}")
print(f"Fine drop: {[f'{d:.3f}' for d in DROP_GRID_FINE]}")

for lr in LR_GRID_FINE:
    for d in DROP_GRID_FINE:
        dh, dl = d, d / 2
        mlp_tmp = MaldiMLPClassifier(
            hidden_dim=512, head_dims=(256, 128), use_attention=False,
            dropout_high=dh, dropout_low=dl, weight_decay=1e-3,
            learning_rate=lr, batch_size=64, epochs=50,
            early_stopping_patience=10, warmup_epochs=10,
            val_fraction=0.1, use_sam=False,
            input_transform="none", tune_threshold=False, random_state=SEED, verbose=False)
        mlp_tmp.fit(X_train, y_train)
        val_balacc = balanced_accuracy_score(y_val, mlp_tmp.predict(X_val))
        mark = " *" if val_balacc > best_balacc else ""
        print(f"  lr={lr:.2e}  drop=({dh:.3f},{dl:.3f})  val_balacc={val_balacc:.4f}{mark}")
        if val_balacc > best_balacc:
            best_balacc = val_balacc; best_lr = lr; best_dh = dh; best_dl = dl

print(f"\nBest after fine: lr={best_lr:.2e}  drop=({best_dh:.3f},{best_dl:.3f})  val_balacc={best_balacc:.4f}")

In [ ]:
# Retrain best + threshold tuning
mlp_reg = MaldiMLPClassifier(
    hidden_dim=512, head_dims=(256, 128), use_attention=False,
    dropout_high=best_dh, dropout_low=best_dl, weight_decay=1e-4,
    learning_rate=best_lr, batch_size=64, epochs=50,
    early_stopping_patience=10, warmup_epochs=10,
    val_fraction=0.1, use_sam=False,
    input_transform="none", tune_threshold=False, random_state=SEED, verbose=True)
mlp_reg.fit(X_train, y_train)

thresholds = np.linspace(0.05, 0.95, 91)
proba_val_b = mlp_reg.predict_proba(X_val)[:, 1]
best_t_b = thresholds[np.argmax([balanced_accuracy_score(y_val, proba_val_b >= t) for t in thresholds])]
preds_b_tuned = (mlp_reg.predict_proba(X_test)[:, 1] >= best_t_b)
test_balacc_b = balanced_accuracy_score(y_test, preds_b_tuned)
test_auc_b = roc_auc_score(y_test, mlp_reg.predict_proba(X_test)[:, 1])
print(f"\nBest MLP: lr={best_lr:.2e}  drop=({best_dh:.3f},{best_dl:.3f})  thresh={best_t_b:.3f}")
print(f"  Test BalAcc (tuned): {test_balacc_b:.4f}  Test AUC: {test_auc_b:.4f}")

### Approach C -- Attention MLP

In [ ]:
# Approach C: Attention MLP
mlp_attn = MaldiMLPClassifier(
    hidden_dim=512, head_dims=(256, 128), use_attention=True,
    dropout_high=0.4, dropout_low=0.2, weight_decay=1e-3,
    learning_rate=1e-3, batch_size=32, epochs=100,
    early_stopping_patience=10, val_fraction=0.1,
    input_transform="none", tune_threshold=False, random_state=SEED, verbose=True)
mlp_attn.fit(X_train, y_train)

proba_val_c = mlp_attn.predict_proba(X_val)[:, 1]
best_t_c = thresholds[np.argmax([balanced_accuracy_score(y_val, proba_val_c >= t) for t in thresholds])]
preds_c_tuned = (mlp_attn.predict_proba(X_test)[:, 1] >= best_t_c)
test_balacc_c = balanced_accuracy_score(y_test, preds_c_tuned)
test_auc_c = roc_auc_score(y_test, mlp_attn.predict_proba(X_test)[:, 1])
print(f"\nAttention MLP  thresh={best_t_c:.3f}")
print(f"  Test BalAcc (tuned): {test_balacc_c:.4f}  Test AUC: {test_auc_c:.4f}")

### Drug 1 Comparison

In [ ]:
# Drug 1 comparison bar chart
labels = ["A: Baseline", "B: Reg (tuned)", "C: Attn (tuned)"]
train_vals = [results_a["train"]["BalAcc"],
              balanced_accuracy_score(y_train, mlp_reg.predict(X_train)),
              balanced_accuracy_score(y_train, mlp_attn.predict(X_train))]
test_vals  = [results_a["test"]["BalAcc"], test_balacc_b, test_balacc_c]
auc_vals   = [results_a["test"]["AUC"], test_auc_b, test_auc_c]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(3); w = 0.25
ax.bar(x - w, train_vals, w, label="Train BalAcc", color="#aec7e8")
ax.bar(x,      test_vals,  w, label="Test BalAcc",  color="#1f77b4")
ax.bar(x + w,  auc_vals,   w, label="Test AUC",     color="#ff7f0e")
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel("Score"); ax.set_title(f"MLP -- {DRUG} (aggregated A+B+C+D)")
ax.legend(loc="lower right"); ax.set_ylim(0, 1.05); ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
for i, (tr, te) in enumerate(zip(train_vals, test_vals)):
    gap = tr - te
    if abs(gap) > 0.005:
        ax.annotate(f"gap={gap:.2f}", (i, (tr+te)/2), fontsize=7, ha="center",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85))
plt.grid(True, ls='--', lw=0.5, color='gray', alpha=0.7)
plt.tight_layout(); plt.savefig(OUT_DIR / "mlp_cip_comparison.pdf")
plt.show()

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(7, 7))
for name, clf, X_eval in [("A: Baseline", mlp_raw, X_test),
                           ("B: Regularised", mlp_reg, X_test),
                           ("C: Attention", mlp_attn, X_test)]:
    RocCurveDisplay.from_predictions(y_test, clf.predict_proba(X_eval)[:, 1], name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_title(f"ROC Curves -- MLP -- {DRUG} (aggregated A+B+C+D)")
plt.tight_layout(); plt.savefig(OUT_DIR / "mlp_cip_roc.pdf")
plt.show()

---
## Drug 2: Amoxicillin-Clavulanic acid (19,967 samples)

In [ ]:
# Drug 2: Amoxicillin-Clavulanic acid
DRUG2 = "Amoxicillin-Clavulanic_acid"
X2_train, y2_train = preprocessed[DRUG2]["train"]
X2_val,   y2_val   = preprocessed[DRUG2]["val"]
X2_test,  y2_test  = preprocessed[DRUG2]["test"]
print(f"\nDrug 2: {DRUG2} -- Train: {X2_train.shape}")

# B: Regularised MLP
mlp2_reg = MaldiMLPClassifier(
    hidden_dim=512, head_dims=(256, 128), use_attention=False,
    dropout_high=best_dh, dropout_low=best_dl, weight_decay=1e-4,
    learning_rate=best_lr, batch_size=64, epochs=50,
    early_stopping_patience=10, warmup_epochs=10,
    val_fraction=0.1, use_sam=False,
    input_transform="none", tune_threshold=False, random_state=SEED, verbose=True)
mlp2_reg.fit(X2_train, y2_train)
proba2_val = mlp2_reg.predict_proba(X2_val)[:, 1]
best_t2 = thresholds[np.argmax([balanced_accuracy_score(y2_val, proba2_val >= t) for t in thresholds])]
preds2b = (mlp2_reg.predict_proba(X2_test)[:, 1] >= best_t2)
bal2b = balanced_accuracy_score(y2_test, preds2b)
auc2b = roc_auc_score(y2_test, mlp2_reg.predict_proba(X2_test)[:, 1])
print(f"B: Reg MLP  thresh={best_t2:.3f}  Test BalAcc={bal2b:.4f}  AUC={auc2b:.4f}")

# C: Attention MLP
mlp2_attn = MaldiMLPClassifier(
    hidden_dim=512, head_dims=(256, 128), use_attention=True,
    dropout_high=0.4, dropout_low=0.2, weight_decay=1e-3,
    learning_rate=1e-3, batch_size=32, epochs=100,
    early_stopping_patience=10, val_fraction=0.1,
    input_transform="none", tune_threshold=False, random_state=SEED, verbose=True)
mlp2_attn.fit(X2_train, y2_train)
proba2c_val = mlp2_attn.predict_proba(X2_val)[:, 1]
best_t2c = thresholds[np.argmax([balanced_accuracy_score(y2_val, proba2c_val >= t) for t in thresholds])]
preds2c = (mlp2_attn.predict_proba(X2_test)[:, 1] >= best_t2c)
bal2c = balanced_accuracy_score(y2_test, preds2c)
auc2c = roc_auc_score(y2_test, mlp2_attn.predict_proba(X2_test)[:, 1])
print(f"C: Attn MLP  thresh={best_t2c:.3f}  Test BalAcc={bal2c:.4f}  AUC={auc2c:.4f}")

---
## Drug 3: Gentamicin (18,312 samples)

In [ ]:
# Drug 3: Gentamicin
DRUG3 = "Gentamicin"
X3_train, y3_train = preprocessed[DRUG3]["train"]
X3_val,   y3_val   = preprocessed[DRUG3]["val"]
X3_test,  y3_test  = preprocessed[DRUG3]["test"]
print(f"\nDrug 3: {DRUG3} -- Train: {X3_train.shape}")

mlp3_reg = MaldiMLPClassifier(
    hidden_dim=512, head_dims=(256, 128), use_attention=False,
    dropout_high=best_dh, dropout_low=best_dl, weight_decay=1e-4,
    learning_rate=best_lr, batch_size=64, epochs=50,
    early_stopping_patience=10, warmup_epochs=10,
    val_fraction=0.1, use_sam=False,
    input_transform="none", tune_threshold=False, random_state=SEED, verbose=True)
mlp3_reg.fit(X3_train, y3_train)
proba3_val = mlp3_reg.predict_proba(X3_val)[:, 1]
best_t3 = thresholds[np.argmax([balanced_accuracy_score(y3_val, proba3_val >= t) for t in thresholds])]
preds3b = (mlp3_reg.predict_proba(X3_test)[:, 1] >= best_t3)
bal3b = balanced_accuracy_score(y3_test, preds3b)
auc3b = roc_auc_score(y3_test, mlp3_reg.predict_proba(X3_test)[:, 1])
print(f"B: Reg MLP  thresh={best_t3:.3f}  Test BalAcc={bal3b:.4f}  AUC={auc3b:.4f}")

mlp3_attn = MaldiMLPClassifier(
    hidden_dim=512, head_dims=(256, 128), use_attention=True,
    dropout_high=0.4, dropout_low=0.2, weight_decay=1e-3,
    learning_rate=1e-3, batch_size=32, epochs=100,
    early_stopping_patience=10, val_fraction=0.1,
    input_transform="none", tune_threshold=False, random_state=SEED, verbose=True)
mlp3_attn.fit(X3_train, y3_train)
proba3c_val = mlp3_attn.predict_proba(X3_val)[:, 1]
best_t3c = thresholds[np.argmax([balanced_accuracy_score(y3_val, proba3c_val >= t) for t in thresholds])]
preds3c = (mlp3_attn.predict_proba(X3_test)[:, 1] >= best_t3c)
bal3c = balanced_accuracy_score(y3_test, preds3c)
auc3c = roc_auc_score(y3_test, mlp3_attn.predict_proba(X3_test)[:, 1])
print(f"C: Attn MLP  thresh={best_t3c:.3f}  Test BalAcc={bal3c:.4f}  AUC={auc3c:.4f}")

---
## Multi-Drug Summary

In [ ]:
# Three-drug summary
multi = pd.DataFrame({
    "Drug": [DRUG, DRUG, DRUG, DRUG2, DRUG2, DRUG3, DRUG3],
    "Approach": ["A: Baseline", "B: Reg (tuned)", "C: Attn (tuned)",
                 "B: Reg (tuned)", "C: Attn (tuned)", "B: Reg (tuned)", "C: Attn (tuned)"],
    "Test BalAcc": [results_a["test"]["BalAcc"], test_balacc_b, test_balacc_c,
                    bal2b, bal2c, bal3b, bal3c],
    "Test AUC": [results_a["test"]["AUC"], test_auc_b, test_auc_c,
                 auc2b, auc2c, auc3b, auc3c],
})
print("\n=== MLP THREE-DRUG SUMMARY (aggregated A+B+C+D) ===")
print(multi.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(7); w = 0.3
ax.bar(x - w/2, multi["Test BalAcc"], w, label="Test BalAcc", color="#1f77b4")
ax.bar(x + w/2, multi["Test AUC"], w, label="Test AUC", color="#ff7f0e")
ax.set_xticks(x)
ax.set_xticklabels([f"{r['Drug'][:10]}\n{r['Approach'][:10]}" for _, r in multi.iterrows()], fontsize=7)
ax.set_ylabel("Score"); ax.set_title("MLP -- Three-Drug Summary (A+B+C+D)")
ax.legend(); ax.set_ylim(0, 1); ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
plt.grid(True, ls='--', lw=0.5, color='gray', alpha=0.7)
plt.tight_layout(); plt.savefig(OUT_DIR / "mlp_three_drug_summary.pdf")
plt.show()

In [ ]:
print("\nDone. MLP results saved to", OUT_DIR.resolve())
for i, drug in enumerate([DRUG, DRUG2, DRUG3], 1):
    _, y, _ = drug_data[drug]
    print(f"  {i}. {drug:35s} {len(y):6d} samples  R={sum(y):5d}  S={len(y)-sum(y):5d}")